# Calling Gemini from Python
In this notebook you will:
1. Load your API key from the `.env` file
2. Choose a model (we use the smallest, cheapest one)
3. Set a **system prompt**, a **user prompt** and a **max output size**
4. Check how many tokens your prompt uses (context size)
5. Call the model and read the reply plus token usage

In [1]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # reads GOOGLE_API_KEY from the .env file
if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError("GOOGLE_API_KEY not found. Add it to your .env file.")

client = genai.Client()  # picks up GOOGLE_API_KEY automatically
print("Client ready")

Client ready


## 1. Pick a model
`gemini-2.5-flash-lite` is Google's smallest and most cost-effective Gemini model. Run the next cell to see other "lite" models you could try.

In [2]:
for m in client.models.list():
    if "lite" in m.name and "tts" not in m.name and "image" not in m.name:
        print(m.name)

models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.5-flash-lite
models/veo-3.1-lite-generate-preview


In [3]:
MODEL = "gemini-2.5-flash-lite"  # change me to try another model

## 2. Prompts and size settings
- **System prompt**: sets the model's role and rules.
- **User prompt**: the actual question or task.
- **Context size**: the model's *context window* (how much it can read) is fixed by the model (about 1M tokens here). You control `MAX_OUTPUT_TOKENS`, which caps the length of the reply.
- **Temperature**: 0 = focused and repeatable, higher = more creative.

In [4]:
SYSTEM_PROMPT = "You are a friendly teacher who explains ideas to beginners in simple words."
USER_PROMPT = "What is a large language model? Explain in 3 short sentences."
MAX_OUTPUT_TOKENS = 300
TEMPERATURE = 0.3

## 3. Check context size
A *token* is roughly 3-4 characters. Let's see how much of the context window your prompt uses.

In [5]:
info = client.models.get(model=MODEL)
count = client.models.count_tokens(model=MODEL, contents=[SYSTEM_PROMPT, USER_PROMPT])
print(f"Model context window : {info.input_token_limit:,} tokens")
print(f"Max output allowed   : {info.output_token_limit:,} tokens")
print(f"Your prompt          : {count.total_tokens:,} tokens")
print(f"Your output cap      : {MAX_OUTPUT_TOKENS:,} tokens")

Model context window : 1,048,576 tokens
Max output allowed   : 65,536 tokens
Your prompt          : 29 tokens
Your output cap      : 300 tokens


## 4. Call the model

In [6]:
response = client.models.generate_content(
    model=MODEL,
    contents=USER_PROMPT,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        temperature=TEMPERATURE,
    ),
)
print(response.text)

Imagine a super-smart computer program that has read tons of books, articles, and websites. It uses all that information to understand and create human-like text. This allows it to answer questions, write stories, and even translate languages!


In [7]:
u = response.usage_metadata
print(f"Prompt tokens   : {u.prompt_token_count}")
print(f"Response tokens : {u.candidates_token_count}")
print(f"Total tokens    : {u.total_token_count}")

Prompt tokens   : 30
Response tokens : 48
Total tokens    : 78


## Try it yourself
- Change the system prompt to a pirate, a lawyer or a poet and re-run section 4.
- Set `MAX_OUTPUT_TOKENS = 20`. What happens to the answer?
- Set `TEMPERATURE = 1.5` and run several times. What changes?
- Swap `MODEL` for another model from the list and compare token usage.